<a href="https://colab.research.google.com/github/kipkii/kipki-s-/blob/main/conveni.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import csv

def load_inventory(filename):
    inventory = {}


    try:
        f = open("/content/products.csv", mode='r', encoding='utf-8-sig')
        f.read() # 테스트로 읽어보기
        f.seek(0) # 커서를 다시 맨 앞으로 이동
    except UnicodeDecodeError:
        f = open("/content/products.csv", mode='r', encoding='cp949')

    with f as file:
        reader = csv.DictReader(file)

        for row in reader:
            p_id = row['product_id']
            # CSV에서 읽어온 데이터는 기본적으로 문자열(String)이므로,
            # 계산이 필요한 가격과 재고는 정수(int)로 변환해 줍니다.
            inventory[p_id] = {
                "category": row['category'],
                "name": row['name'],
                "price": int(row['price']),
                "stock": int(row['stock']),
                "order_date" : row['order_date'],
                "expiration_date": row['expiration_date']
            }

    return inventory

# 함수 실행하여 인벤토리 데이터 불러오기
inventory = load_inventory('products.csv')

# 제대로 불러왔는지 확인해보기 (첫 3개 상품만 출력)
print("✅ 인벤토리 로드 완료!\n")
count = 0
for p_id, info in inventory.items():
    print(f"[{p_id}] {info['category']} - {info['name']} (가격: {info['price']}원, 재고: {info['stock']}개)")
    count += 1
    if count == 3:
        break

✅ 인벤토리 로드 완료!

[P001] 도시락 - 백종원 제육도시락 (가격: 3000원, 재고: 21개)
[P002] 도시락 - 11찬 정찬도시락 (가격: 4500원, 재고: 21개)
[P003] 도시락 - 돈까스 도시락 (가격: 1500원, 재고: 26개)


In [3]:
def shopping_process(inventory):
    cart = {} # 장바구니 딕셔너리

    # 1. 인벤토리에서 대분류(category)만 뽑아서 중복 제거 후 리스트로 만듦
    categories = list(set(item['category'] for item in inventory.values()))
    categories.sort() # 가나다순으로 보기 좋게 정렬

    while True:
        print("\n========== [편의점 POS 시스템] ==========")
        for i, cat in enumerate(categories, 1):
            print(f"{i}. {cat}")
        print("0. 결제 단계로 넘어가기")
        print("=========================================")

        cat_choice = input("원하시는 카테고리를 입력하세요: ")

        # 0을 누르면 쇼핑 종료하고 결제 단계로 넘어감
        if cat_choice == '0':
            break

        # 예외 처리: 숫자가 아니거나 메뉴 범위를 벗어난 경우
        if not cat_choice.isdigit() or not (1 <= int(cat_choice) <= len(categories)):
            print("❌ 잘못된 입력입니다. 다시 선택해주세요.")
            continue

        selected_category = categories[int(cat_choice) - 1]
        print(f"\n--- [{selected_category}] 상품 목록 ---")

        # 2. 선택한 카테고리의 상품만 필터링해서 보여주기
        category_products = {}
        for p_id, info in inventory.items():
            if info['category'] == selected_category:
                category_products[p_id] = info
                print(f"ID: {p_id} | {info['name']} | 가격: {info['price']}원 | 재고: {info['stock']}개")
        print("-" * 35)

        # 3. 상품 선택 및 수량 입력
        p_choice = input("구매할 상품 ID를 입력하세요 (이전 메뉴는 엔터키): ").strip().upper()

        # 엔터키를 치면 다시 대분류 선택으로 돌아감
        if not p_choice:
            continue

        # 예외 처리: 카테고리에 없는 ID를 입력한 경우
        if p_choice not in category_products:
            print("❌ 해당 카테고리에 없는 상품 ID입니다.")
            continue

        qty_input = input(f"'{category_products[p_choice]['name']}' 구매 수량을 입력하세요: ")

        # 예외 처리: 수량을 잘못 입력한 경우
        if not qty_input.isdigit() or int(qty_input) <= 0:
            print("❌ 수량은 1 이상의 숫자로 입력해야 합니다.")
            continue

        buy_qty = int(qty_input)
        current_stock = category_products[p_choice]['stock']

        # 장바구니에 이미 담긴 수량까지 고려해서 재고 체크
        already_in_cart = cart.get(p_choice, {}).get('qty', 0)

        if buy_qty + already_in_cart > current_stock:
            print(f"❌ 재고가 부족합니다! (현재 남은 재고: {current_stock - already_in_cart}개)")
            continue

        # 4. 장바구니(Dictionary)에 담기
        if p_choice in cart:
            cart[p_choice]['qty'] += buy_qty
            cart[p_choice]['subtotal'] += buy_qty * category_products[p_choice]['price']
        else:
            cart[p_choice] = {
                "name": category_products[p_choice]['name'],
                "price": category_products[p_choice]['price'],
                "qty": buy_qty,
                "subtotal": buy_qty * category_products[p_choice]['price']
            }

        print(f"✅ 장바구니 담기 완료: {category_products[p_choice]['name']} {buy_qty}개")

    return cart

# 함수 실행하여 장바구니 데이터 받아오기
my_cart = shopping_process(inventory)

# 장바구니 결과 확인
print("\n[현재 장바구니에 담긴 내역]")
print(my_cart)


========== [편의점 POS 시스템] ==========
1. 과자/스낵
2. 김밥/주먹밥
3. 냉장/냉동식품
4. 도시락
5. 라면/면류
6. 샌드위치/햄버거
7. 아이스크림
8. 유제품
9. 음료
10. 일상용품
0. 결제 단계로 넘어가기
원하시는 카테고리를 입력하세요: 1

--- [과자/스낵] 상품 목록 ---
ID: P051 | 포카칩 | 가격: 1200원 | 재고: 27개
ID: P052 | 스윙칩 | 가격: 1700원 | 재고: 24개
ID: P053 | 새우깡 | 가격: 5000원 | 재고: 22개
ID: P054 | 홈런볼 | 가격: 1700원 | 재고: 20개
ID: P055 | 꼬북칩 | 가격: 1000원 | 재고: 27개
ID: P056 | 프링글스 | 가격: 1200원 | 재고: 26개
ID: P057 | 맛동산 | 가격: 3000원 | 재고: 23개
ID: P058 | 오징어땅콩 | 가격: 5000원 | 재고: 27개
ID: P059 | 빼빼로 | 가격: 1200원 | 재고: 20개
ID: P060 | 초코파이 | 가격: 4500원 | 재고: 22개
-----------------------------------
구매할 상품 ID를 입력하세요 (이전 메뉴는 엔터키): P051
'포카칩' 구매 수량을 입력하세요: 2
✅ 장바구니 담기 완료: 포카칩 2개

========== [편의점 POS 시스템] ==========
1. 과자/스낵
2. 김밥/주먹밥
3. 냉장/냉동식품
4. 도시락
5. 라면/면류
6. 샌드위치/햄버거
7. 아이스크림
8. 유제품
9. 음료
10. 일상용품
0. 결제 단계로 넘어가기
원하시는 카테고리를 입력하세요: 1

--- [과자/스낵] 상품 목록 ---
ID: P051 | 포카칩 | 가격: 1200원 | 재고: 27개
ID: P052 | 스윙칩 | 가격: 1700원 | 재고: 24개
ID: P053 | 새우깡 | 가격: 5000원 | 재고: 22개
ID: P054 | 홈런볼 | 가격: 1700원 | 재고:

KeyboardInterrupt: Interrupted by user

In [4]:
import csv
from datetime import datetime, timedelta

def load_inventory(filename):
    """CSV 파일에서 인벤토리 데이터를 읽어와 딕셔너리로 반환합니다."""
    inventory = {}
    try:
        f = open(filename, mode='r', encoding='utf-8-sig')
        f.read()
        f.seek(0)
    except UnicodeDecodeError:
        f = open(filename, mode='r', encoding='cp949')

    with f as file:
        reader = csv.DictReader(file)
        for row in reader:
            inventory[row['product_id']] = {
                "category": row['category'],
                "name": row['name'],
                "price": int(row['price']),
                "stock": int(row['stock']),
                "expiration_date": row['expiration_date']
            }
    return inventory

def run_pos_system():
    # 1. 초기 데이터 및 변수 세팅
    inventory = load_inventory('products.csv')
    total_revenue = 0  # 총 누적 매출액
    pending_orders = [] # 발주 대기열 리스트

    # 2. 메인 루프 (고객 1명의 결제 사이클)
    while True:
        cart = {} # 장바구니 (요구사항: 딕셔너리 형태로 받아옴)
        categories = sorted(list(set(item['category'] for item in inventory.values())))

        print("\n" + "="*40)
        print("편의점 결제 시스템을 시작합니다.")
        print("="*40)

        # --- [상품 담기 단계] ---
        while True:
            print("\n[대분류 목록]")
            for i, cat in enumerate(categories, 1):
                print(f"{i}. {cat}")
            print("0. 결제 단계로 넘어가기")

            cat_choice = input("\n원하시는 카테고리 번호를 입력하세요: ")

            if cat_choice == '0':
                if not cart:
                    print("장바구니가 비어있습니다. 상품을 담아주세요.")
                    continue
                break # 결제 단계로 이동

            if not cat_choice.isdigit() or not (1 <= int(cat_choice) <= len(categories)):
                print("잘못된 입력입니다.")
                continue

            selected_category = categories[int(cat_choice) - 1]
            print(f"\n--- [{selected_category}] 상품 목록 ---")

            # 해당 카테고리 상품 노출
            category_products = {}
            for p_id, info in inventory.items():
                if info['category'] == selected_category:
                    category_products[p_id] = info
                    print(f"[{p_id}] {info['name']} | {info['price']}원 | 남은재고: {info['stock']}개")

            p_choice = input("\n구매할 상품의 ID를 입력하세요 (이전 메뉴는 엔터키): ").strip().upper()
            if not p_choice:
                continue

            if p_choice not in category_products:
                print("유효하지 않은 상품 ID입니다.")
                continue

            qty_input = input(f"'{category_products[p_choice]['name']}' 구매 수량을 입력하세요: ")
            if not qty_input.isdigit() or int(qty_input) <= 0:
                print("수량은 1 이상의 숫자로 입력해주세요.")
                continue

            buy_qty = int(qty_input)

            # 재고 검증 (장바구니에 담긴 수량 포함)
            already_in_cart = cart.get(p_choice, {}).get('qty', 0)
            if buy_qty + already_in_cart > category_products[p_choice]['stock']:
                print("재고가 부족합니다.")
                continue

            # 장바구니 딕셔너리에 추가
            if p_choice in cart:
                cart[p_choice]['qty'] += buy_qty
                cart[p_choice]['subtotal'] += buy_qty * category_products[p_choice]['price']
            else:
                cart[p_choice] = {
                    "name": category_products[p_choice]['name'],
                    "price": category_products[p_choice]['price'],
                    "qty": buy_qty,
                    "subtotal": buy_qty * category_products[p_choice]['price']
                }
            print(f"장바구니에 담겼습니다! (현재 {cart[p_choice]['qty']}개)")

        # --- [결제 및 영수증 출력 단계] ---
        total_amount = sum(item['subtotal'] for item in cart.values())
        total_revenue += total_amount # 매출로 잡힘

        print("\n" + "="*15 + " 영수증 " + "="*15)
        for p_id, info in cart.items():
            print(f"- {info['name']} | {info['qty']}개 | {info['subtotal']}원")
        print("-" * 38)
        print(f"총 결제 금액: {total_amount}원")
        print("="*38 + "\n")

        # --- [재고 차감 및 자동 발주 단계] ---
        # 테스트용 코드: 내가 원하는 특정 날짜로 세팅 (예: 2026년 6월 1일 오후 2시)
        today = datetime(2026, 6, 6, 14, 0, 0)
        #today = datetime.now()

        for p_id, info in cart.items():
            # 1. 인벤토리에서 차감
            inventory[p_id]['stock'] -= info['qty']
            current_stock = inventory[p_id]['stock']

            # 2. 10개 이하일 시 발주 로직
            if current_stock <= 10:
                print(f"🚨 [발주 필요] '{info['name']}'의 재고가 {current_stock}개 남았습니다.")

                order_qty = 20
                restock_date = today + timedelta(days=1)
                expiration_date = today + timedelta(days=7)

                # 발주 내역 기록
                pending_orders.append({
                    "product_id": p_id,
                    "name": info['name'],
                    "order_qty": order_qty,
                    "order_date": today.strftime("%Y-%m-%d"),
                    "restock_date": restock_date.strftime("%Y-%m-%d"),
                    "new_expiration": expiration_date.strftime("%Y-%m-%d")
                })
                print(f"{order_qty}개 자동 발주 완료! (입고예정: {restock_date.strftime('%Y-%m-%d')}, 유통기한: {expiration_date.strftime('%Y-%m-%d')})")

        print(f"\n현재까지 총 누적 매출: {total_revenue}원")

        # 다음 손님을 받을지 종료할지 선택
        next_step = input("\n다음 결제를 진행하시겠습니까? (Y/N): ").strip().upper()
        if next_step != 'Y':
            print("\n시스템을 종료합니다. 수고하셨습니다!")
            break

# 프로그램 실행
if __name__ == "__main__":
    run_pos_system()


🏪 편의점 결제 시스템을 시작합니다.

[대분류 목록]
1. 과자/스낵
2. 김밥/주먹밥
3. 냉장/냉동식품
4. 도시락
5. 라면/면류
6. 샌드위치/햄버거
7. 아이스크림
8. 유제품
9. 음료
10. 일상용품
0. 결제 단계로 넘어가기

원하시는 카테고리 번호를 입력하세요: 1

--- [과자/스낵] 상품 목록 ---
[P051] 포카칩 | 1200원 | 남은재고: 27개
[P052] 스윙칩 | 1700원 | 남은재고: 24개
[P053] 새우깡 | 5000원 | 남은재고: 22개
[P054] 홈런볼 | 1700원 | 남은재고: 20개
[P055] 꼬북칩 | 1000원 | 남은재고: 27개
[P056] 프링글스 | 1200원 | 남은재고: 26개
[P057] 맛동산 | 3000원 | 남은재고: 23개
[P058] 오징어땅콩 | 5000원 | 남은재고: 27개
[P059] 빼빼로 | 1200원 | 남은재고: 20개
[P060] 초코파이 | 4500원 | 남은재고: 22개

구매할 상품의 ID를 입력하세요 (이전 메뉴는 엔터키): P051
'포카칩' 구매 수량을 입력하세요: 3
✅ 장바구니에 담겼습니다! (현재 3개)

[대분류 목록]
1. 과자/스낵
2. 김밥/주먹밥
3. 냉장/냉동식품
4. 도시락
5. 라면/면류
6. 샌드위치/햄버거
7. 아이스크림
8. 유제품
9. 음료
10. 일상용품
0. 결제 단계로 넘어가기

원하시는 카테고리 번호를 입력하세요: 0

=============== 영수증 ===============
- 포카칩 | 3개 | 3600원
--------------------------------------
총 결제 금액: 3600원


현재까지 총 누적 매출: 3600원

다음 결제를 진행하시겠습니까? (Y/N): Y

🏪 편의점 결제 시스템을 시작합니다.

[대분류 목록]
1. 과자/스낵
2. 김밥/주먹밥
3. 냉장/냉동식품
4. 도시락
5. 라면/면류
6. 샌드위치/햄버거
7. 아이스크림
8. 유제품
9. 음료
10. 일상용품
0.

KeyboardInterrupt: Interrupted by user